In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score
)
from imblearn.metrics import geometric_mean_score


In [6]:
LABEL_COL = "label_3"
CLASS_NAMES = ["Average", "Good", "Exellent"]

EPOCHS = 30
BATCH_SIZE = 64

def load_csv(path):
    df = pd.read_csv(path)
    df = df.dropna()
    return df

train_df = load_csv("/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/train_median_cdsmote.csv")
val_df   = load_csv("/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/val.csv")

test_files = {
    "test_1": "/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/test_1.csv",
    "test_2": "/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/test_2.csv",
    "test_3": "/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/test_3.csv",
    "test_4": "/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_cdsmote/test_4.csv",
}


In [7]:
X_train = train_df.drop(columns=[LABEL_COL]).values
y_train = train_df[LABEL_COL].values

X_val = val_df.drop(columns=[LABEL_COL]).values
y_val = val_df[LABEL_COL].values

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Adjust X_train to match the number of features in X_val by removing the first column (Unnamed: 0)
# This assumes 'Unnamed: 0' is the first column and is not present in X_val
# A more robust solution would be to handle this during data loading or in the previous cell.
if X_train.shape[1] > X_val.shape[1]:
    X_train = X_train[:, 1:]

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

In [9]:
# reshape: (samples, timesteps, features=1)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val   = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)

TIMESTEPS = X_train.shape[1]

model = Sequential([
    Bidirectional(LSTM(64, return_sequences=False),
                  input_shape=(TIMESTEPS, 1)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        33,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,243 (165.01 KB)

 Trainable params: 42,243 (165.01 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)


Epoch 1/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 91s 18ms/step - accuracy: 0.7362 - loss: 0.5586 - val_accuracy: 0.0742 - val_loss: 6.1072
Epoch 2/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9237 - loss: 0.2025 - val_accuracy: 0.2285 - val_loss: 3.7716
Epoch 3/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9515 - loss: 0.1364 - val_accuracy: 0.1716 - val_loss: 3.6013
Epoch 4/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 85s 18ms/step - accuracy: 0.9638 - loss: 0.1046 - val_accuracy: 0.0786 - val_loss: 5.5326
Epoch 5/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9694 - loss: 0.0888 - val_accuracy: 0.1011 - val_loss: 5.6042
Epoch 6/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9764 - loss: 0.0710 - val_accuracy: 0.0662 - val_loss: 7.2871
Epoch 7/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9799 - loss: 0.0610 - val_accuracy: 0.0724 - val_loss: 6.1520
Epoch 8/10
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 86s 18ms/step - accuracy: 0.9814 -

In [11]:
def evaluate_metrics(y_true, y_pred):
    res = {}

    res["Accuracy"] = accuracy_score(y_true, y_pred)
    res["BalancedAcc"] = balanced_accuracy_score(y_true, y_pred)

    res["Precision Macro"] = precision_score(y_true, y_pred, average="macro")
    res["Precision Weighted"] = precision_score(y_true, y_pred, average="weighted")

    res["Recall Macro"] = recall_score(y_true, y_pred, average="macro")
    res["Recall Weighted"] = recall_score(y_true, y_pred, average="weighted")

    res["F1-Score Macro"] = f1_score(y_true, y_pred, average="macro")
    res["F1-Score Weighted"] = f1_score(y_true, y_pred, average="weighted")

    res["GMean"] = geometric_mean_score(y_true, y_pred, average="macro")
    res["MCC"] = matthews_corrcoef(y_true, y_pred)
    res["Kappa"] = cohen_kappa_score(y_true, y_pred)

    for i, name in enumerate(CLASS_NAMES):
        yt = (y_true == i).astype(int)
        yp = (y_pred == i).astype(int)

        res[f"Label ({name}) Precision"] = precision_score(yt, yp)
        res[f"Label ({name}) Recall"] = recall_score(yt, yp)
        res[f"Label ({name}) F1-Score"] = f1_score(yt, yp)
        res[f"Label ({name}) G-Mean"] = geometric_mean_score(yt, yp)

    return res


In [12]:
final_results = {}
for test_name, path in test_files.items():
    df_test = load_csv(path)

    X_test = df_test.drop(columns=[LABEL_COL]).values
    y_test = df_test[LABEL_COL].values

    X_test = scaler.transform(X_test)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    y_prob = model.predict(X_test)
    y_pred = np.argmax(y_prob, axis=1)

    metrics = evaluate_metrics(y_test, y_pred)

    for k, v in metrics.items():
        final_results[f"{test_name}_{k}"] = v
result_df = pd.DataFrame([final_results])
result_df.to_csv("BiLSTM_Tabular_4Test_Metrics.csv", index=False)

result_df


7265/7265 ━━━━━━━━━━━━━━━━━━━━ 46s 6ms/step
7265/7265 ━━━━━━━━━━━━━━━━━━━━ 46s 6ms/step
7265/7265 ━━━━━━━━━━━━━━━━━━━━ 46s 6ms/step
7265/7265 ━━━━━━━━━━━━━━━━━━━━ 46s 6ms/step


,test_1_Accuracy,test_1_BalancedAcc,test_1_Precision Macro,test_1_Precision Weighted,test_1_Recall Macro,test_1_Recall Weighted,test_1_F1-Score Macro,test_1_F1-Score Weighted,test_1_GMean,test_1_MCC,...,test_4_Label (Average) F1-Score,test_4_Label (Average) G-Mean,test_4_Label (Good) Precision,test_4_Label (Good) Recall,test_4_Label (Good) F1-Score,test_4_Label (Good) G-Mean,test_4_Label (Exellent) Precision,test_4_Label (Exellent) Recall,test_4_Label (Exellent) F1-Score,test_4_Label (Exellent) G-Mean
0,0.210593,0.489394,0.350078,0.996388,0.489394,0.210593,0.144639,0.344047,0.599417,0.028671,...,0.262172,0.559978,0.002797,0.965289,0.005578,0.313739,1.0,0.100818,0.183169,0.317519
